# Cigna RAG Demo 2 (BAAI/bge-m3) — LangGraph 파이프라인

**SKN24 3차 프로젝트 · 4팀 · 담당: 김은우**

| 기능 | 참고 강사 코드 |
|------|----------------|
| HyDE (가상 문서 생성 검색) | `02-4_hyde.ipynb` |
| 문서 관련성 평가 (Grade) | `07_agentic_rag.ipynb` |
| 쿼리 재작성 (Rewrite) | `07_agentic_rag.ipynb` |
| LangGraph 상태 관리 | `07_agentic_rag.ipynb`, `04_chatbot_with_memory.ipynb` |
| BM25+Dense+RRF 하이브리드 | `02-2_bm25_dense_retrieval.ipynb`, `02-3_rrf.ipynb` |

## Section 0 · 환경 설정

In [ ]:
# !pip install -q langchain langchain-community langchain-openai langchain-chroma
# !pip install -q chromadb openai tiktoken
# !pip install -q pypdf pdfplumber rank_bm25
# !pip install -q langgraph  # LangGraph 추가
# !pip install -q sentence-transformers langchain-huggingface
print('✅ 설치 완료')

✅ 설치 완료


In [ ]:
import os, json, re
from pathlib import Path
from typing import List, Dict, Any, Literal, Optional
from dotenv import load_dotenv

BASE_DIR = Path('.')
CIGNA_DIR = BASE_DIR / 'Cigna'

for sub in ['Customer_Guide', 'Policy_Rules', 'Benefits_Summary']:
    p = CIGNA_DIR / sub
    files = list(p.glob('*.pdf')) if p.exists() else []
    print(f'{sub}: {len(files)} PDFs')

load_dotenv()

Customer_Guide: 4 PDFs
Policy_Rules: 5 PDFs
Benefits_Summary: 3 PDFs


True

---
## Section 1 · 데이터 로드

### 수정 사항
- `clean_table_row`: 헤더 행의 공백 셀을 `Not covered`로 오판하던 버그 수정 → `has_monetary_value()`로 데이터 행 여부 판단
- `format_multicurrency`: 다통화 셀 (`$25,000\n€18,500\n£16,500`) → `USD $25,000 / EUR €18,500 / GBP £16,500`
- `load_pdf`: Customer Guide + Policy Rules도 pdfplumber로 통일 (표 파싱 일관성)

In [ ]:
import pdfplumber
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHECKMARK_GLYPHS = {'\uf0fc', '\uf0b7', '✓', '✔'}

def has_monetary_value(row) -> bool:
    """행에 금액·퍼센트 값이 있으면 데이터 행으로 판단 (헤더 행 공백 오판 방지)."""
    for cell in row:
        if cell and any(c in str(cell) for c in ['$', '€', '£', '%', 'Paid in full', 'N/A']):
            return True
    return False

def format_multicurrency(text: str) -> str:
    """다통화 셀: '$25,000\n€18,500\n£16,500' → 'USD $25,000 / EUR €18,500 / GBP £16,500'"""
    currency_map = {'$': 'USD', '€': 'EUR', '£': 'GBP'}
    parts = [p.strip() for p in text.split('\n') if p.strip()]
    if len(parts) <= 1:
        return text
    labeled = []
    for part in parts:
        for sym, label in currency_map.items():
            if part.startswith(sym):
                labeled.append(f'{label} {part}')
                break
        else:
            labeled.append(part)
    return ' / '.join(labeled)

def clean_table_row(row: list) -> str:
    """표 행 정제: 체크마크→Covered, 데이터행 빈셀→Not covered, 다통화 포맷."""
    is_data_row = has_monetary_value(row)
    cleaned = []
    for cell in row:
        if cell is None:
            continue
        cell_str = str(cell).strip()
        # 회전된 섹션 라벨 제거 (전부 대문자 + 6자 이상)
        if cell_str.replace('\n','').isupper() and len(cell_str.replace('\n','')) > 6:
            continue
        has_ck = any(g in cell_str for g in CHECKMARK_GLYPHS)
        remaining = cell_str
        for g in CHECKMARK_GLYPHS:
            remaining = remaining.replace(g, '').strip()
        if has_ck:
            cleaned.append(f'Covered - {remaining}' if remaining else 'Covered')
        elif cell_str == '':
            if is_data_row:           # ← 데이터 행의 빈 셀만 Not covered
                cleaned.append('Not covered')
            # 헤더 행 빈 셀은 무시
        elif cell_str:
            cleaned.append(format_multicurrency(cell_str))
    return ' | '.join(cleaned)



# -- 마크다운 표 변환 함수 (Section 1-B-2, load_pdf에서도 사용) --
import re

_CHECK = {'\uf0fc', '\uf0b7', '\u2713', '\u2714', '\u2611', '\u2705'}
_CROSS = {'\u2297', '\u2717', '\u2718', '\u274c', '\u00d7', '\u2612', '\u29bb', '\uf078'}
_BADGE = re.compile(r'^(Updated|New\b|\d+\s*MONTHS?)', re.IGNORECASE)


def _cvt(cell, is_data):
    if cell is None or not str(cell).strip():
        return 'Not Covered' if is_data else ''
    t = str(cell).strip().replace('\n', ' ')
    chk = any(g in t for g in _CHECK)
    crs = any(g in t for g in _CROSS)
    clean = t
    for g in _CHECK | _CROSS:
        clean = clean.replace(g, '')
    clean = clean.strip()
    if crs:
        return 'Not Covered'
    if chk:
        return f"Covered{' - ' + clean if clean else ''}"
    return clean or ('Not Covered' if is_data else '')


def _is_data(row):
    return any(
        c and re.search(r'[$\u20ac\xa3\d]|covered|paid|n/a|refund|no coverage', str(c), re.I)
        for c in row
    )


def _is_rotated(text):
    t = (text or '').strip()
    return bool(t) and '\n' in t and t.replace('\n', '').replace(' ', '').isupper() and len(t) > 6


def _clean_benefit(text, extra_badges=()):
    if not text and not extra_badges:
        return ''
    parts = [p.strip() for p in re.split(r'[\n/]+', text or '') if p.strip()]
    if len(parts) <= 1:
        badges  = list(extra_badges)
        content = [text.strip()] if text and text.strip() else []
    else:
        badges  = [p for p in parts if _BADGE.match(p)] + list(extra_badges)
        content = [p for p in parts if not _BADGE.match(p)]
    result = ' '.join(content)
    if badges:
        result = f"{result} ({' '.join(badges)})" if result else f"({' '.join(badges)})"
    return result.strip()


def _col_map(table):
    s = g = p = hdr = None
    for i, row in enumerate(table[:8]):
        v = [str(c or '').split('\n')[0].strip() for c in row]
        if 'Silver' in v and 'Gold' in v:
            s, g = v.index('Silver'), v.index('Gold')
            p = next((j for j, x in enumerate(v) if x == 'Platinum'), None)
            hdr = i
            break
    if s is None:
        return None
    b = max(0, s - 1)
    for row in table[hdr + 1:]:
        if not _is_data(row):
            continue
        for ci in range(s - 1, -1, -1):
            v = str(row[ci] or '').strip()
            if v and not _is_rotated(v):
                b = ci
                break
        break
    return {'b': b, 's': s, 'g': g, 'p': p}


def _table_to_md(table):
    cm = _col_map(table)
    lines = []
    hdr_done = False
    for row in table:
        data = _is_data(row)
        if cm:
            n     = len(row)
            b_raw = str(row[cm['b']] or '').strip() if cm['b'] < n else ''
            mid_badges = []
            for ci in range(cm['b'] + 1, cm['s']):
                v = str(row[ci] or '').strip()
                if v and _BADGE.match(v):
                    mid_badges.append(v)
                elif v:
                    b_raw = f'{b_raw} {v}'.strip()
            s_raw = row[cm['s']] if cm['s'] < n else None
            g_raw = row[cm['g']] if cm['g'] < n else None
            p_raw = row[cm['p']] if cm['p'] and cm['p'] < n else None
            if not any(str(v or '').strip() for v in [s_raw, g_raw, p_raw]):
                data = False
            b_txt = _clean_benefit(b_raw, mid_badges)
            cells = [b_txt, _cvt(s_raw, data), _cvt(g_raw, data)]
            if cm['p']:
                cells.append(_cvt(p_raw, data))
        else:
            cells = [_cvt(c, data) for c in row]
        if not any(str(c).strip() for c in cells):
            continue
        line = '| ' + ' | '.join(str(c) for c in cells) + ' |'
        lines.append(line)
        if not hdr_done:
            lines.append('| ' + ' | '.join(['---'] * len(cells)) + ' |')
            hdr_done = True
    return '\n'.join(lines)


print('clean_table_row / _table_to_md 준비 완료')


clean_table_row / _table_to_md 준비 완료


In [ ]:
PDF_META = [
    {'path':'Cigna/Customer_Guide/200008 CGHO Customer Guide EN_05_2019.pdf',
     'source_type':'customer_guide','doc_version':'2019','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Customer_Guide/591048 CGHO Customer Guide EN_05_2022.pdf',
     'source_type':'customer_guide','doc_version':'2022','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Customer_Guide/591048-cgho-customer-guide-en_05_2023.pdf',
     'source_type':'customer_guide','doc_version':'2023','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Customer_Guide/Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf',
     'source_type':'customer_guide','doc_version':'2026','is_latest':True,'plan_type':'all'},
    {'path':'Cigna/Policy_Rules/200008 CGHO Customer Guide EN_05_2019.pdf',
     'source_type':'policy_rules','doc_version':'2019','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Policy_Rules/CGHO Policy Rules CGIC NA_EN_05_2023.pdf',
     'source_type':'policy_rules','doc_version':'2023','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Policy_Rules/CGHO Policy Rules CGIC_EN_02_2024.pdf',
     'source_type':'policy_rules','doc_version':'2024','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Policy_Rules/CGHO Policy Rules CGIC_EN_02_2025.pdf',
     'source_type':'policy_rules','doc_version':'2025','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Policy_Rules/CGHP Policy Rules CGIC EN 02_2026.pdf',
     'source_type':'policy_rules','doc_version':'2026','is_latest':True,'plan_type':'all'},
    {'path':'Cigna/Benefits_Summary/591116 Cigna_Global_International_Health_Plans_Benefits_Summary_USD_EN_0523.pdf',
     'source_type':'benefits_summary','doc_version':'2023-05','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Benefits_Summary/591116 Cigna Global Benefits Summary USD_EN_0924.pdf',
     'source_type':'benefits_summary','doc_version':'2024-09','is_latest':False,'plan_type':'all'},
    {'path':'Cigna/Benefits_Summary/591116-cigna-global-benefits-summary-usd_en_02_2025.pdf',
     'source_type':'benefits_summary','doc_version':'2025-02','is_latest':True,'plan_type':'all'},
]
print(f'총 {len(PDF_META)}개 PDF 정의')

총 12개 PDF 정의


In [ ]:
def load_pdf(meta: dict) -> List[Document]:
    """모든 PDF 통합 로더 — pdfplumber로 텍스트+표 동시 추출."""
    docs = []
    try:
        with pdfplumber.open(meta['path']) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ''
                tables = page.extract_tables()
                # _table_to_md: 마크다운 표 형식 (Cell 5에 정의)
                # Benefits_Summary 회전 라벨 / Customer_Guide 빈 스페이서 자동 처리
                table_mds = [md for t in tables if (md := _table_to_md(t))]
                combined = text
                if table_mds:
                    combined += '\n\n[TABLE]\n' + '\n\n'.join(table_mds)
                if combined.strip():
                    docs.append(Document(
                        page_content=combined,
                        metadata={**meta,'page':page_num,'file_name':Path(meta['path']).name}
                    ))
    except Exception as e:
        print(f'  ⚠ 오류 {meta["path"]}: {e}')
    return docs

all_raw_docs: List[Document] = []
for meta in PDF_META:
    path = Path(meta['path'])
    if not path.exists():
        print(f'  ⚠ 파일 없음: {path}')
        continue
    docs = load_pdf(meta)
    all_raw_docs.extend(docs)
    print(f'  ✅ [{meta["source_type"]} {meta["doc_version"]}] {len(docs)} pages')
print(f'\n총 {len(all_raw_docs)} pages')

  ✅ [customer_guide 2019] 44 pages
  ✅ [customer_guide 2022] 43 pages
  ✅ [customer_guide 2023] 44 pages
  ✅ [customer_guide 2026] 56 pages
  ✅ [policy_rules 2019] 44 pages
  ✅ [policy_rules 2023] 18 pages
  ✅ [policy_rules 2024] 20 pages
  ✅ [policy_rules 2025] 22 pages
  ✅ [policy_rules 2026] 26 pages
  ✅ [benefits_summary 2023-05] 5 pages
  ✅ [benefits_summary 2024-09] 5 pages
  ✅ [benefits_summary 2025-02] 5 pages

총 332 pages


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100,
    separators=['\n\n','\n','. ',' ',''],
)
all_chunks: List[Document] = text_splitter.split_documents(all_raw_docs)
print(f'청크 수: {len(all_chunks)}')

청크 수: 3438


---
## Section 1-B · PDF 파싱 품질 검증

모든 PDF를 실제로 열어 **표 파싱 품질**을 다각도로 검증합니다.  
`load_pdf()` / `clean_table_row()` 실행 후 바로 사용 가능합니다. (API 키 불필요)


In [ ]:
# ════════════════════════════════════════════════════════════════
# Section 1-B · PDF 파싱 품질 검증
# API 키 불필요 — Section 1 실행 직후 바로 실행 가능
# ════════════════════════════════════════════════════════════════
from pathlib import Path

SEP = '─' * 65

# ── 1) 전체 파싱 실행 및 통계 수집 ────────────────────────────────
parse_stats   = []   # (meta, pages, table_rows, raw_tables)
all_raw_docs  = []

print('\n▶ PDF 로드 중...\n')
for meta in PDF_META:
    abs_path = Path(meta['path'])
    if not abs_path.exists():
        print(f'  ✗ 파일 없음: {abs_path.name}')
        parse_stats.append((meta, 0, [], []))
        continue

    pages_ok, table_rows, raw_tables = 0, [], []
    try:
        import pdfplumber
        with pdfplumber.open(str(abs_path)) as pdf:
            for pn, page in enumerate(pdf.pages, 1):
                text   = page.extract_text() or ''
                tables = page.extract_tables()
                has_content = bool(text.strip())
                for ti, tbl in enumerate(tables):
                    raw_tables.append((pn, ti, tbl))
                    has_content = True
                    for row in tbl:
                        c = clean_table_row(row)
                        if c:
                            table_rows.append((pn, c))
                if has_content:
                    pages_ok += 1
    except Exception as e:
        print(f'  ✗ 오류: {e}')

    latest_mark = ' ★' if meta['is_latest'] else ''
    print(f'  ✓ [{meta["source_type"]} {meta["doc_version"]}]{latest_mark}'
          f'  {pages_ok}p  표행={len(table_rows)}  원시표={len(raw_tables)}')
    parse_stats.append((meta, pages_ok, table_rows, raw_tables))

# ── 2) 요약 표 ────────────────────────────────────────────────────
print(f'\n{SEP}')
print(f'{"source_type":<22}{"ver":<10}{"★":<4}{"pages":>6}{"표행수":>8}{"원시표":>8}')
print(SEP)
for meta, pages, trows, rtables in parse_stats:
    print(f'{meta["source_type"]:<22}{meta["doc_version"]:<10}'
          f'{"★" if meta["is_latest"] else "":<4}'
          f'{pages:>6}{len(trows):>8}{len(rtables):>8}')

# ── 3) 이상 탐지 (최신 문서만) ────────────────────────────────────
print(f'\n{SEP}')
print('  이상 탐지 (is_latest=True 문서)')
print(SEP)

for meta, pages, trows, rtables in parse_stats:
    if not meta['is_latest'] or not trows:
        continue
    texts = [t for _, t in trows]
    nc_cnt  = sum(1 for t in texts if 'Not covered' in t)
    ck_cnt  = sum(1 for t in texts if t.startswith('Covered'))
    mc_cnt  = sum(1 for t in texts if 'USD' in t or 'EUR' in t)
    only_nc = [t for t in texts if t.strip() == 'Not covered']

    print(f'\n  [{meta["source_type"]} {meta["doc_version"]}]')
    print(f'    전체 정제 행  : {len(texts)}')
    print(f'    Not covered   : {nc_cnt} ({nc_cnt/len(texts)*100:.1f}%)')
    print(f'    Covered 변환  : {ck_cnt}')
    print(f'    다통화(USD/€) : {mc_cnt}')
    warn = ' ⚠ 과다 의심' if len(only_nc) > 5 else ' ✓'
    print(f'    단독 Not cov  : {len(only_nc)}{warn}')

# ── 4) source_type별 첫 번째 표 샘플 (최신 문서) ─────────────────
print(f'\n{SEP}')
print('  표 샘플 — 최신 문서별 첫 번째 표 (최대 6행 · raw→정제 비교)')
print(SEP)

shown = set()
for meta, pages, trows, rtables in parse_stats:
    stype = meta['source_type']
    if not meta['is_latest'] or stype in shown or not rtables:
        continue
    shown.add(stype)
    pn, ti, raw_tbl = rtables[0]
    print(f'\n  ▶ {stype} ({meta["doc_version"]}) — p.{pn} 표#{ti+1} / 총 {len(raw_tbl)}행')
    for ri, row in enumerate(raw_tbl[:6]):
        raw_repr = [str(c)[:28].replace('\n','↵') if c else '∅' for c in row]
        cleaned  = clean_table_row(row)
        print(f'    행{ri+1} raw : {raw_repr}')
        print(f'    행{ri+1} 정제: {cleaned or "(빈 행 — 무시됨)"}')
        print()

# ── 5) 키워드 검색 (최신 문서) ────────────────────────────────────
KEYWORDS = ['Deductible','Out-of-Pocket','Private room',
            'Newborn','Maternity','Emergency','Dental','Vision','Mental']

print(f'{SEP}')
print('  키워드 검색 — 최신 문서 한정')
print(SEP)

for meta, pages, trows, rtables in parse_stats:
    if not meta['is_latest'] or not trows:
        continue
    texts = [t for _, t in trows]
    found = {kw: [t for t in texts if kw.lower() in t.lower()] for kw in KEYWORDS}
    found = {k:v for k,v in found.items() if v}
    print(f'\n  [{meta["source_type"]} {meta["doc_version"]}]')
    if not found:
        print('    (키워드 해당 없음)')
    for kw, hits in found.items():
        print(f'    "{kw}" → {len(hits)}개')
        for h in hits[:2]:
            print(f'      예) {h[:90]}')

# ── 6) Benefits Summary 표 전체 출력 (최신) ──────────────────────
print(f'\n{SEP}')
print('  Benefits Summary 최신 표 행 전체 (최대 40행)')
print(SEP)

for meta, pages, trows, rtables in parse_stats:
    if meta['source_type'] != 'benefits_summary' or not meta['is_latest']:
        continue
    print(f'  버전: {meta["doc_version"]} / 총 {len(trows)}행\n')
    for i, (pg, row_text) in enumerate(trows[:40]):
        print(f'  p{pg:>3} │ {row_text}')
    if len(trows) > 40:
        print(f'  ... 외 {len(trows)-40}행 생략')

print(f'\n{"═"*65}')
print('  PDF 파싱 검증 완료')
print('═'*65)



▶ PDF 로드 중...

  ✓ [customer_guide 2019]  44p  표행=221  원시표=75
  ✓ [customer_guide 2022]  43p  표행=226  원시표=87
  ✓ [customer_guide 2023]  44p  표행=237  원시표=89
  ✓ [customer_guide 2026] ★  56p  표행=238  원시표=92
  ✓ [policy_rules 2019]  44p  표행=221  원시표=75
  ✓ [policy_rules 2023]  18p  표행=13  원시표=11
  ✓ [policy_rules 2024]  20p  표행=14  원시표=11
  ✓ [policy_rules 2025]  22p  표행=14  원시표=11
  ✓ [policy_rules 2026] ★  26p  표행=13  원시표=11
  ✗ 파일 없음: 591116 Cigna_Global_International_Health_Plans_Benefits_Summary_USD_EN_0523.pdf
  ✗ 파일 없음: 591116 Cigna Global Benefits Summary USD_EN_0924.pdf
  ✗ 파일 없음: 591116-cigna-global-benefits-summary-usd_en_02_2025.pdf

─────────────────────────────────────────────────────────────────
source_type           ver       ★    pages     표행수     원시표
─────────────────────────────────────────────────────────────────
customer_guide        2019              44     221      75
customer_guide        2022              43     226      87
customer_guide        2023             

---
## Section 1-B-2 · pdfplumber → 마크다운 표 변환

pdfplumber로 추출한 표를 **마크다운 테이블** 형식으로 변환합니다.  
LlamaParse와 동일한 문제들을 함께 해결합니다.

| 문제 | 해결 |
|------|------|
| `Updated` / `12 MONTHS` 배지가 Silver 콼럼에 들어감 | 항목명으로 병합 |
| 긴 설명이 Silver 콼럼으로 밀림 | 항목명으로 병합 |
| `✓` / `` 체크마크 | `Covered` 로 변환 |
| `⊗` X 동그라미 | `Not Covered` 로 변환 |
| 데이터 행 빈 셀 | `Not Covered` 로 변환 |
| 헤더 행 빈 셀 | 무시 |


In [ ]:
# =========================================================
# Section 1-B-2 · pdfplumber -> 마크다운 표 변환 (검증용)
# _table_to_md 등 핵심 함수는 Section 1 (Cell 5)에 정의됨
# load_pdf()도 동일 함수를 사용합니다
# =========================================================


def pdf_to_markdown(pdf_path, pages=None):
    """PDF -> 마크다운 표 (검증/확인용).
    pages=None : 전체, pages=[2,3] : 특정 페이지 (1-indexed)
    """
    out = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        target = (
            [pdf.pages[i - 1] for i in pages if 1 <= i <= len(pdf.pages)]
            if pages else pdf.pages
        )
        for pg in target:
            pn     = pg.page_number
            tables = pg.extract_tables()
            text   = pg.extract_text() or ''
            out.append(f'\n## p.{pn}\n')
            if text.strip() and not tables:
                out.append(text.strip())
            for ti, tbl in enumerate(tables):
                md = _table_to_md(tbl)
                if md:
                    out.append(f'\n<!-- 표 #{ti + 1} -->\n{md}')
    return '\n'.join(out)


# -- 실행 ---------------------------------------------------
PDF_BENEFITS = next(Path('cigna/Benefits_Summary').glob('*2025*.pdf'), None)
PDF_GUIDE    = next(Path('cigna/Customer_Guide').glob('*2026*.pdf'), None)

if PDF_BENEFITS:
    print('=== Benefits_Summary p.2 ===')
    print(pdf_to_markdown(PDF_BENEFITS, pages=[2]))
    print('\n=== Benefits_Summary p.3 ===')
    print(pdf_to_markdown(PDF_BENEFITS, pages=[3]))

if PDF_GUIDE:
    print('\n=== Customer_Guide p.17~19 ===')
    print(pdf_to_markdown(PDF_GUIDE, pages=[17, 18, 19]))


In [ ]:
# 확인하기2

# ── Customer Guide / Benefits Summary 각 1페이지 전체 파싱 확인 ──
# Section 1 실행 후 아무 셀에나 붙여넣어 실행 (노트북 수정 없음)

import pdfplumber
from pathlib import Path

def show_page(pdf_path: str, page_num: int):
    """지정 페이지의 텍스트 + 표를 그대로 출력."""
    path = Path(pdf_path)
    print(f"\n{'═'*65}")
    print(f"  파일: {path.name}  /  p.{page_num}")
    print('═'*65)

    with pdfplumber.open(str(path)) as pdf:
        if page_num > len(pdf.pages):
            print(f"  ⚠ 페이지 없음 (전체 {len(pdf.pages)}p)")
            return

        page   = pdf.pages[page_num - 1]
        text   = page.extract_text() or ''
        tables = page.extract_tables()

        # ── 1. 본문 텍스트 ────────────────────────────────────────
        print("\n[본문 텍스트]")
        if text.strip():
            print(text)
        else:
            print("  (텍스트 없음)")

        # ── 2. 표 (raw → 정제 비교) ──────────────────────────────
        if not tables:
            print("\n[표 없음]")
            return

        for ti, table in enumerate(tables):
            print(f"\n[표 #{ti+1}  /  {len(table)}행 × {len(table[0]) if table else 0}열]")
            print(f"{'행':>3}  {'raw':^55}  {'정제 결과'}")
            print('─'*90)
            for ri, row in enumerate(table):
                raw_repr = [str(c)[:20].replace('\n','↵') if c else '∅' for c in row]
                cleaned  = clean_table_row(row)  # Section 1의 함수 사용
                print(f"  {ri+1:>2}  {str(raw_repr):<55}  {cleaned}")

# ─── 확인할 페이지 지정 ─────────────────────────────────────────────
# 표가 많은 페이지 번호를 직접 바꿔가며 확인하세요

# Customer Guide 최신 (2026) — 표 많은 페이지 예: 10
show_page(
    'data_cigna/Customer_Guide/Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf',
    page_num=19
)

# Benefits Summary 최신 (2025-02) — 표 많은 페이지 예: 3
show_page(
    'data_cigna/Benefits_Summary/591116-cigna-global-benefits-summary-usd_en_02_2025.pdf',
    page_num=3
)


═════════════════════════════════════════════════════════════════
  파일: Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf  /  p.19
═════════════════════════════════════════════════════════════════

[본문 텍스트]
Silver Gold Platinum
Accident and Emergency Room treatment
Up to the total limit shown for your selected plan per $500 $1,000 $2,000
€370 €740 €1,600
beneficiary per period of cover.
£335 £665 £1,300
We will pay for necessary emergency treatment that is required on an outpatient basis only at an Accident and
Emergency department in a hospital following an accident, sudden illness, and/or life threatening situations, and where
the beneficiary does not occupy a bed overnight for medical reasons.
Important notes:
• If you have selected the International Outpatient option; this benefit and the limits are satisfied first and then the
applicable International Outpatient benefits can be used thereafter.
• No deductible or cost share that you may have selected on the International Med

---
## Section 1-C · LlamaParse — 마크다운 형식 표 추출

`pdfplumber`의 한계(표 열 관계 소실, 체크마크 누락)를 보완하기 위해  
**LlamaParse**(LlamaIndex Cloud)로 PDF를 마크다운 형식으로 파싱합니다.

- 표를 `| col1 | col2 |` 형태의 마크다운 테이블로 자동 변환
- LLM이 열 관계를 명확히 파악 가능
- [API 키 발급](https://cloud.llamaindex.ai/) → 환경변수 `LLAMA_CLOUD_API_KEY`


In [ ]:
# !pip install llama-parse llama-index

In [ ]:
# ════════════════════════════════════════════════════════════════
# Section 1-C · LlamaParse — 마크다운 형식 표 추출
# 필요 패키지: pip install llama-parse
# API 키: https://cloud.llamaindex.ai/ 에서 발급 후 환경변수 설정
# ════════════════════════════════════════════════════════════════

# !pip install -q llama-parse

import os
from pathlib import Path

# ── 설정 ──────────────────────────────────────────────────────────
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")

# 검증할 PDF (최신 버전 각 1개)
TARGET_PDFS = {
    "Benefits_Summary": "cigna/Benefits_Summary/591116-cigna-global-benefits-summary-usd_en_02_2025.pdf",
    "Customer_Guide":   "cigna/Customer_Guide/Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf",
}

# ── LlamaParse 초기화 ─────────────────────────────────────────────
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=LLAMA_CLOUD_API_KEY,
    result_type="markdown",          # 표를 | col | 형식 마크다운으로 추출
    verbose=False,
    language="en",
    skip_diagonal_text=True,         # 회전 텍스트(섹션 라벨) 무시
)

# ── 파싱 실행 + 결과 출력 ──────────────────────────────────────────
parsed_results = {}   # {pdf_type: [page_text, ...]}

for pdf_type, pdf_path in TARGET_PDFS.items():
    abs_path = Path(pdf_path)
    if not abs_path.exists():
        print(f"  ✗ 파일 없음: {abs_path.name}")
        continue

    print(f"  ⏳ 파싱 중: {abs_path.name} ...")
    try:
        docs = parser.load_data(str(abs_path))   # List[Document], 페이지별
        parsed_results[pdf_type] = docs
        print(f"  ✓ {pdf_type}: {len(docs)}페이지 파싱 완료")
    except Exception as e:
        print(f"  ✗ 오류: {e}")

# ── 페이지별 마크다운 출력 ─────────────────────────────────────────
SEP = "═" * 65

for pdf_type, docs in parsed_results.items():
    print(f"\n{SEP}")
    print(f"  {pdf_type}  —  총 {len(docs)}페이지")
    print(SEP)

    for page_idx, doc in enumerate(docs[:5], start=1):   # 처음 5페이지만
        text = doc.text.strip()
        if not text:
            continue

        # 표가 있는 페이지만 상세 출력 (마크다운 테이블 감지)
        has_table = "|" in text and "---" in text
        print(f"\n── p.{page_idx}  {'[표 포함]' if has_table else '[텍스트만]'} ──")
        print(text[:2000])   # 너무 길면 앞 2000자만

        if has_table:
            # 표 행만 추출해서 따로 보여주기
            table_lines = [l for l in text.split("\n") if l.strip().startswith("|")]
            if table_lines:
                print(f"\n  → 추출된 마크다운 표 ({len(table_lines)}행):")
                for line in table_lines[:20]:   # 최대 20행
                    print(f"    {line}")



  ✗ 파일 없음: 591116-cigna-global-benefits-summary-usd_en_02_2025.pdf
  ✗ 파일 없음: Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf


/var/folders/sk/_9bn_nr9727cxqwf9sxykqxr0000gn/T/ipykernel_80458/370405880.py:22: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [ ]:
# ── pdfplumber 와 같은 페이지 비교 ─────────────────────────────────
import pdfplumber

print(f"\n{SEP}")
print("  비교: 같은 페이지를 pdfplumber vs LlamaParse 로 비교")
print(SEP)

COMPARE_PAGE = 3   # ← 비교할 페이지 번호 (바꿔가며 확인)
compare_path = TARGET_PDFS.get("Benefits_Summary", "")

if compare_path and Path(compare_path).exists():
    # pdfplumber
    with pdfplumber.open(compare_path) as pdf:
        page = pdf.pages[COMPARE_PAGE - 1]
        raw_text  = page.extract_text() or ""
        raw_tables = page.extract_tables()
        table_rows = []
        for tbl in raw_tables:
            for row in tbl:
                cleaned = clean_table_row(row)   # Section 1 함수 재사용
                if cleaned:
                    table_rows.append(cleaned)

    print(f"\n[pdfplumber  — p.{COMPARE_PAGE}]")
    print("  extract_text():")
    print(f"  {raw_text[:300].replace(chr(10), ' ')}")
    if table_rows:
        print("  extract_tables() → clean_table_row():")
        for row in table_rows[:8]:
            print(f"    {row}")

    # LlamaParse
    if "Benefits_Summary" in parsed_results and len(parsed_results["Benefits_Summary"]) >= COMPARE_PAGE:
        llama_page = parsed_results["Benefits_Summary"][COMPARE_PAGE - 1].text
        print(f"\n[LlamaParse  — p.{COMPARE_PAGE}]")
        table_lines = [l for l in llama_page.split("\n") if l.strip().startswith("|")]
        if table_lines:
            print(f"  마크다운 표 ({len(table_lines)}행):")
            for line in table_lines[:10]:
                print(f"    {line}")
        else:
            print(f"  {llama_page[:400]}")

print(f"\n{SEP}")
print("  Section 1-C 완료")
print(SEP)



═════════════════════════════════════════════════════════════════
  비교: 같은 페이지를 pdfplumber vs LlamaParse 로 비교
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  Section 1-C 완료
═════════════════════════════════════════════════════════════════


---
### Section 1-C-2 · LlamaParse 출력 후처리

LlamaParse 출력의 3가지 핵심 문제를 후처리로 수정합니다.

| 문제 | 원인 | 해결 방법 |
|------|------|-----------|
| `Updated` / `12 MONTHS` 가 Silver 컬럼에 들어감 | PDF 배지가 별도 셀로 분리됨 | 항목명으로 병합 후 값 컬럼 재정렬 |
| 긴 설명이 Silver 컬럼에 들어감 | 줄 바꿈 설명이 첫 번째 값 컬럼으로 밀려 들어감 | 항목명으로 병합 |
| 체크/X 기호 드롭 → 컬럼 밀림 | LlamaParse 가 기호를 텍스트로 변환 안 함 | `parsing_instruction` 으로 소스에서 수정 |


In [ ]:
# ════════════════════════════════════════════════════════════════
# Section 1-C-2 · LlamaParse 출력 후처리
# 기존 Section 1-C 코드는 유지하고, 그 출력에 후처리를 적용합니다.
# ════════════════════════════════════════════════════════════════

import re
from pathlib import Path

# ── 1. parsing_instruction 으로 소스 단계에서 체크/X 처리 ──────────────
PARSING_INSTRUCTION = (
    "This is an insurance benefits table.\n"
    "Column order: Benefit | Silver | Gold | Platinum | Close Care\n"
    "CRITICAL rules:\n"
    "1. Checkmark cells (tick symbol) -> write the word Covered.\n"
    "2. Circled-X cells (blocked/crossed symbol) -> write Not Covered.\n"
    "3. Updated is a badge on the benefit name row. "
    "   Append (Updated) to the benefit name. Do NOT put it in Silver column.\n"
    "4. Time labels like 12 MONTHS or 24 MONTHS are waiting period badges. "
    "   Append (12 MONTHS waiting) to the benefit name. Do NOT place in Silver.\n"
    "5. Descriptions starting with We will pay for, Up to a maximum of, "
    "   Rental only, Including, Per night belong to the benefit name column.\n"
    "6. Always output exactly 4 value columns per benefit row.\n"
)

from llama_parse import LlamaParse

improved_parser = LlamaParse(
    api_key=LLAMA_CLOUD_API_KEY,
    result_type="markdown",
    verbose=False,
    language="en",
    skip_diagonal_text=True,
    parsing_instruction=PARSING_INSTRUCTION,
)

# ── 2. 후처리 함수 (잔여 케이스용) ────────────────────────────────────
BADGE_RE = re.compile(
    r"^(Updated|"
    r"\d+\s*MONTHS?|"
    r"We will pay for[:\s]|"
    r"Up to a (maximum|combined maximum)|"
    r"Including (physical|inpatient|internal)|"
    r"Rental only|Per night|Access to|24/7|"
    r"Available to beneficiaries)",
    re.IGNORECASE
)

CHECK_CHARS = set("\u2713\u2714\uf0fc\uf0b7\u2611\u2705")
CROSS_CHARS = set("\u2297\u2717\u2718\u274c\u00d7\u2612")

def normalize_cell(text):
    t = text.strip()
    if t and all(c in CHECK_CHARS for c in t):
        return "Covered"
    if t and all(c in CROSS_CHARS for c in t):
        return "Not Covered"
    return t

def fix_row(row):
    row = row.strip()
    if not row.startswith("|") or re.match(r"\|\s*[-:]+", row):
        return row
    parts = [p.strip() for p in row.split("|")]
    if parts and parts[0] == "": parts = parts[1:]
    if parts and parts[-1] == "": parts = parts[:-1]
    if len(parts) < 2:
        return row
    benefit_col = parts[0]
    absorbed, remaining = [], []
    for cell in parts[1:]:
        if BADGE_RE.match(cell):
            absorbed.append(cell)
        else:
            remaining.append(normalize_cell(cell))
    if absorbed:
        benefit_col = benefit_col + " (" + " / ".join(absorbed) + ")"
    return "| " + " | ".join([benefit_col] + remaining) + " |"

def postprocess_markdown(md_text):
    return "\n".join(
        fix_row(line) if line.strip().startswith("|") else line
        for line in md_text.split("\n")
    )

print("fix_row / postprocess_markdown 준비 완료")

# ── 3. Before / After 비교 (API 없이 즉시 확인) ──────────────────────
SAMPLE_ROWS = [
    ("Consultations - Updated 배지",
     "| Consultations with medical practitioners | Updated | $2,500 | $7,500 | $650 | |"),
    ("Pre-natal - 12 MONTHS 배지",
     "| Pre-natal and post natal care | 12 MONTHS | $3,500 | $7,000 | | |"),
    ("Outpatient Rehab - 긴 설명이 Silver 컬럼으로 밀림",
     "| Outpatient Rehabilitation | We will pay for: Physiotherapy $1,000. | | | | |"),
    ("Genetic Testing - 12 MONTHS + Updated 중복 배지",
     "| Genetic Testing | 12 MONTHS | Updated | $1,000 | $2,000 | $4,000 |"),
    ("Infertility - 긴 설명",
     "| Infertility Investigations | Up to a maximum of $10,000. Available to beneficiaries. | | | | |"),
    ("Hormone Replacement - Updated 배지",
     "| Hormone Replacement Therapy | Updated | $500 | $1,000 | $1,500 | |"),
    ("Treatment for Obesity - 24 MONTHS 배지",
     "| Treatment for Obesity | 24 MONTHS | $20,000 | $25,000 | | |"),
]

SEP = "=" * 65
print(f"\n{SEP}")
print("  Before / After 비교")
print(SEP)
for label, row in SAMPLE_ROWS:
    fixed = fix_row(row)
    icon = "OK" if row.strip() != fixed.strip() else "WARN(변경없음)"
    print(f"\n[{icon}] {label}")
    print(f"  before: {row[:95]}")
    print(f"  after:  {fixed[:95]}")
print(f"\n{SEP}")

# ── 4. 개선된 파서로 재파싱 + 후처리 (API 호출) ─────────────────────
BENEFITS_PATH = "cigna/Benefits_Summary/591116-cigna-global-benefits-summary-usd_en_02_2025.pdf"

if Path(BENEFITS_PATH).exists():
    print("\nimproved_parser 로 재파싱 중...")
    try:
        docs_improved = improved_parser.load_data(BENEFITS_PATH)
        print(f"  {len(docs_improved)}페이지 완료")
        for page_idx, doc in enumerate(docs_improved[:5], start=1):
            fixed_text = postprocess_markdown(doc.text)
            data_rows = [
                l for l in fixed_text.split("\n")
                if l.strip().startswith("|") and not re.match(r"\|\s*[-:]+", l)
            ]
            if not data_rows: continue
            print(f"\n{SEP}  p.{page_idx} — {len(data_rows)}행\n")
            for r in data_rows[:12]:
                print(f"  {r}")
            if len(data_rows) > 12:
                print(f"  ... 외 {len(data_rows)-12}행")
    except Exception as e:
        print(f"  오류: {e}")
else:
    print(f"  파일 없음: {BENEFITS_PATH}")

print(f"\n{SEP}")
print("  Section 1-C-2 완료")
print(SEP)


fix_row / postprocess_markdown 준비 완료

  Before / After 비교

[OK] Consultations - Updated 배지
  before: | Consultations with medical practitioners | Updated | $2,500 | $7,500 | $650 | |
  after:  | Consultations with medical practitioners (Updated) | $2,500 | $7,500 | $650 |  |

[OK] Pre-natal - 12 MONTHS 배지
  before: | Pre-natal and post natal care | 12 MONTHS | $3,500 | $7,000 | | |
  after:  | Pre-natal and post natal care (12 MONTHS) | $3,500 | $7,000 |  |  |

[OK] Outpatient Rehab - 긴 설명이 Silver 컬럼으로 밀림
  before: | Outpatient Rehabilitation | We will pay for: Physiotherapy $1,000. | | | | |
  after:  | Outpatient Rehabilitation (We will pay for: Physiotherapy $1,000.) |  |  |  |  |

[OK] Genetic Testing - 12 MONTHS + Updated 중복 배지
  before: | Genetic Testing | 12 MONTHS | Updated | $1,000 | $2,000 | $4,000 |
  after:  | Genetic Testing (12 MONTHS / Updated) | $1,000 | $2,000 | $4,000 |

[OK] Infertility - 긴 설명
  before: | Infertility Investigations | Up to a maximum of $10,000. Avail

---
## Section 2 · 임베딩 + 벡터스토어

In [ ]:
# !pip install -U langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# BAAI/bge-m3: 100개 이상 언어 지원 다국어 임베딩 모델 (dense 768-dim)
# normalize_embeddings=True → 코사인 유사도 바로 사용 가능
embed_baai = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},       # GPU 사용 시 'cuda' 로 변경
    encode_kwargs={'normalize_embeddings': True},
)

latest_chunks = [c for c in all_chunks if c.metadata.get('is_latest', False)]
print(f'최신 버전 청크: {len(latest_chunks)} / 전체: {len(all_chunks)}')

vectorstore_latest = Chroma.from_documents(
    documents=latest_chunks, embedding=embed_baai,
    collection_name='cigna_latest', persist_directory='./chroma_baai_latest',
)
vectorstore_all = Chroma.from_documents(
    documents=all_chunks, embedding=embed_baai,
    collection_name='cigna_all', persist_directory='./chroma_baai_all',
)
print('✅ 벡터스토어 구축 완료')

최신 버전 청크: 897 / 전체: 3438


---
## Section 2-A · 코사인 유사도 헬퍼

질문 ↔ 답변 간 의미적 유사도를 측정합니다.  
값이 **0.7 이상**이면 답변이 질문에 잘 대응한다고 볼 수 있습니다.


In [ ]:
# ════════════════════════════════════════════════════════════════
# Section 2-A · 코사인 유사도 헬퍼
# embed_baai 가 정의된 뒤에 실행하세요
# ════════════════════════════════════════════════════════════════
import numpy as np

def cosine_sim(text1: str, text2: str) -> float:
    """
    두 텍스트 간 코사인 유사도 (BAAI/bge-m3).
    반환값: 0.0 ~ 1.0
    """
    if not text1.strip() or not text2.strip():
        return 0.0
    vecs = embed_baai.embed_documents([text1, text2])
    a, b = np.array(vecs[0]), np.array(vecs[1])
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def print_sim(question: str, answer: str, label: str = '') -> float:
    """유사도 계산 후 출력 + 반환."""
    sim = cosine_sim(question, answer)
    bar_len = int(sim * 20)
    bar = '█' * bar_len + '░' * (20 - bar_len)
    tag = label + ' ' if label else ''
    level = '🟢 우수' if sim >= 0.75 else ('🟡 보통' if sim >= 0.55 else '🔴 낮음')
    print(f'  {tag}질문↔답변 코사인 유사도: {sim:.4f}  [{bar}]  {level}')
    return sim

print('cosine_sim / print_sim 준비 완료')


cosine_sim / print_sim 준비 완료


---
## Section 2-B · 벡터스토어 업데이트 (신규 문서 교체)

새로운 PDF가 나왔을 때: 기존 동일 `source_type`의 최신 문서를 내리고 새 문서를 올린다.

In [ ]:
def update_vectorstore_latest(new_pdf_meta: dict) -> None:
    """
    신규 PDF가 나왔을 때 vectorstore_latest를 업데이트한다.
    1. 같은 source_type의 기존 최신 문서 ID를 조회해 삭제
    2. 새 PDF를 로드·청킹 후 추가
    3. PDF_META 리스트도 갱신
    """
    new_type = new_pdf_meta['source_type']

    # ── Step 1: 기존 latest 문서 삭제 ──────────────
    existing = vectorstore_latest.get(
        where={'$and': [{'source_type': {'$eq': new_type}},
                        {'is_latest': {'$eq': True}}]}
    )
    old_ids = existing['ids']
    if old_ids:
        vectorstore_latest.delete(ids=old_ids)
        print(f'  🗑 기존 최신 청크 {len(old_ids)}개 삭제 ({new_type})')

    # ── Step 2: PDF_META 갱신 ─────────────────────
    for m in PDF_META:
        if m['source_type'] == new_type and m['is_latest']:
            m['is_latest'] = False  # 이전 최신 → False
    new_pdf_meta['is_latest'] = True
    PDF_META.append(new_pdf_meta)

    # ── Step 3: 새 문서 로드 + 청킹 + 추가 ────────
    new_raw = load_pdf(new_pdf_meta)
    new_chunks = text_splitter.split_documents(new_raw)
    vectorstore_latest.add_documents(new_chunks)
    print(f'  ✅ 신규 최신 청크 {len(new_chunks)}개 추가 ({new_type} {new_pdf_meta["doc_version"]})')

# ── 사용 예시 (실제 파일 있을 때 주석 해제) ──
# new_meta = {
#     'path': 'Cigna/Customer_Guide/Cigna-Customer-Guide_2027.pdf',
#     'source_type': 'customer_guide',
#     'doc_version': '2027',
#     'is_latest': True,
#     'plan_type': 'all',
# }
# update_vectorstore_latest(new_meta)
print('update_vectorstore_latest() 준비 완료')

update_vectorstore_latest() 준비 완료
